# VIT Pipeline — Dataset Analysis
Run after Stage 4 (quality filter) to inspect your dataset.

In [ ]:
import json, sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from src.utils import load_jsonl

records = load_jsonl('../data/filtered/filtered.jsonl')
df = pd.DataFrame([{
    'type':         r.get('type'),
    'source':       r.get('source'),
    'clip_score':   r.get('clip_score', 0),
    'h_score':      r.get('h_score', 1),
    'resp_words':   len(r['response'].split()),
    'instr_words':  len(r['instruction'].split()),
} for r in records])
print(f'Total samples: {len(df)}')
df.head()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. Instruction type distribution
df['type'].value_counts().plot.bar(ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Instruction type distribution')
axes[0,0].set_xlabel('')

# 2. Source distribution
df['source'].value_counts().plot.pie(ax=axes[0,1], autopct='%1.0f%%')
axes[0,1].set_title('Image source distribution')

# 3. CLIP score distribution
df['clip_score'].hist(ax=axes[0,2], bins=40, color='coral', edgecolor='white')
axes[0,2].set_title('CLIP score distribution')
axes[0,2].axvline(df['clip_score'].mean(), color='red', linestyle='--', label=f'Mean {df["clip_score"].mean():.3f}')
axes[0,2].legend()

# 4. Response length by type
df.groupby('type')['resp_words'].mean().sort_values().plot.barh(ax=axes[1,0], color='teal')
axes[1,0].set_title('Avg response length by type (words)')

# 5. CLIP score by type
df.groupby('type')['clip_score'].mean().sort_values().plot.barh(ax=axes[1,1], color='purple')
axes[1,1].set_title('Avg CLIP score by type')

# 6. Response length histogram
df['resp_words'].hist(ax=axes[1,2], bins=40, color='gold', edgecolor='white')
axes[1,2].set_title('Response length distribution (words)')

plt.tight_layout()
plt.savefig('../results/dataset_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Spot-check a few samples per type
from PIL import Image
import matplotlib.pyplot as plt
import random

for itype in df['type'].unique():
    sample = df[df['type'] == itype].sample(1).index[0]
    rec = records[sample]
    
    fig, ax = plt.subplots(1, 1, figsize=(5, 4))
    try:
        img = Image.open(rec['image'])
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'[{itype}] CLIP={rec.get("clip_score",0):.3f}', fontsize=10)
    except:
        ax.text(0.5, 0.5, 'Image not found', ha='center')
    plt.tight_layout()
    plt.show()
    
    print(f'Q: {rec["instruction"]}')
    print(f'A: {rec["response"]}')
    print('-'*60)

In [ ]:
# Benchmark results comparison table
# Fill in after running lmms-eval on both baseline and fine-tuned model
import json
results = {
    'Baseline (LLaVA-1.6 zero-shot)': {'MMBench': None, 'POPE_acc': None, 'MME_P': None},
    'Fine-tuned (ours)':              {'MMBench': None, 'POPE_acc': None, 'MME_P': None},
}
# After running eval, load from results JSON:
# with open('../results/lmms_eval/merged/results.json') as f:
#     raw = json.load(f)
# results['Fine-tuned (ours)']['MMBench'] = raw['results']['mmbench_en']['mmbench_overall,none']

pd.DataFrame(results).T